In [ ]:
# ==============================================================================
# Reproducibility Note: To comply with strict reproducibility standards, a fixed
# random seed (random_state=42) is globally enforced across all operations.
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

# Enforce global seed for absolute determinism
np.random.seed(42)

In [ ]:
# ==============================================================================
# Step 1: Data Loading & SHFSF Feature Selection (Manuscript Section 4.1.4)
# ==============================================================================
# Using a relative path so it runs universally on any machine
data_path = 'My dataset with class and without missing values.csv'
print(f"Loading harmonized dataset from '{data_path}'...")
df = pd.read_csv(data_path)

# Isolate the Top 6 features selected by the SHFSF consensus mechanism
shfsf_features = ['mag', 'magType', 'Year', 'gap', 'magError', 'depth']
X = df[shfsf_features]
y = df['mag class']

# One-hot encode the categorical 'magType' feature
X = pd.get_dummies(X, columns=['magType'], drop_first=True)

# Encode the target severity labels
le = LabelEncoder()
y = le.fit_transform(y)

Loading harmonized dataset from 'My dataset with class and without missing values.csv'...


In [ ]:
# ==============================================================================
# Step 2: Stratified Data Splitting & Scaling (Manuscript Section 3.3 & 4.1.3)
# ==============================================================================
# 70:30 Stratified split using the fixed random seed
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Standardize features (Scaling parameters derived ONLY from training data)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train_scaled.shape}")
print(f"Testing set shape: {X_test_scaled.shape}")
print("-" * 50)

Training set shape: (13581, 12)
Testing set shape: (5821, 12)
--------------------------------------------------


In [ ]:
# ==============================================================================
# Step 3: Define Hyperparameter Search Grids (Aligning with Section 4.1.3)
# ==============================================================================
# The search spaces are designed to validate the optimal configurations reported
# in the final manuscript via 5-fold cross-validation.
param_grids = {
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),
        'params': {
            'penalty': ['l2'], # Corrected from '12' to 'l2' (Letter L)
            'C': [0.1, 1.0, 10.0]
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'params': {
            'criterion': ['gini', 'entropy'],
            'max_depth': [None, 10, 20, 50]
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20]
        }
    },
    'SVM': {
        'model': SVC(random_state=42),
        'params': {
            'kernel': ['linear', 'rbf'],
            'C': [0.1, 1, 10],
            'gamma': ['scale', 'auto']
        }
    },
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': [3, 5, 7, 9],
            'metric': ['euclidean', 'manhattan']
        }
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100, 150, 200],
            'learning_rate': [0.1, 0.5, 0.8, 1.0]
        }
    },
    'XGBoost': {
        # Removed use_label_encoder=False as it is deprecated in newer XGBoost versions
        'model': XGBClassifier(random_state=42, eval_metric='mlogloss'),
        'params': {
            'n_estimators': [100, 200, 250],
            'max_depth': [3, 6, 9],
            'learning_rate': [0.01, 0.05, 0.1]
        }
    }
}


In [ ]:
# ==============================================================================
# Step 4: Execute Grid Search Cross-Validation
# ==============================================================================
print("Executing 5-Fold GridSearchCV across all models...\n")
best_estimators = {}

for model_name, config in param_grids.items():
    print(f"Optimizing {model_name}...")

    # Initialize GridSearchCV with 5 folds
    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        cv=5,
        scoring='accuracy',
        n_jobs=-1
    )

    # Fit the grid search on the scaled training data
    grid_search.fit(X_train_scaled, y_train)

    # Store the best model
    best_estimators[model_name] = grid_search.best_estimator_

    # Print the discovered optimal parameters
    print(f"Optimal Parameters for {model_name}:")
    print(grid_search.best_params_)
    print(f"Best 5-Fold CV Accuracy: {grid_search.best_score_:.4f}\n")

print("=" * 50)
print("Hyperparameter tuning complete.")

Executing 5-Fold GridSearchCV across all models...

Optimizing Logistic Regression...
Optimal Parameters for Logistic Regression:
{'C': 10.0, 'penalty': 'l2'}
Best 5-Fold CV Accuracy: 0.9993

Optimizing Decision Tree...
Optimal Parameters for Decision Tree:
{'criterion': 'gini', 'max_depth': None}
Best 5-Fold CV Accuracy: 0.9998

Optimizing Random Forest...
Optimal Parameters for Random Forest:
{'max_depth': None, 'n_estimators': 300}
Best 5-Fold CV Accuracy: 0.9990

Optimizing SVM...
Optimal Parameters for SVM:
{'C': 10, 'gamma': 'scale', 'kernel': 'linear'}
Best 5-Fold CV Accuracy: 0.9993

Optimizing KNN...
Optimal Parameters for KNN:
{'metric': 'manhattan', 'n_neighbors': 3}
Best 5-Fold CV Accuracy: 0.9901

Optimizing AdaBoost...
Optimal Parameters for AdaBoost:
{'learning_rate': 0.5, 'n_estimators': 50}
Best 5-Fold CV Accuracy: 0.9984

Optimizing XGBoost...
Optimal Parameters for XGBoost:
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200}
Best 5-Fold CV Accuracy: 1.0000

